# Task 1 — Forward Kinematics of the BayMax

---

This notebook presents the derivation process of the **forward kinematics (FK)** of the BayMax 5-DOF robotic arm, from first principles (Homogeneous Matrix Transform), and using SymPy python package for symbolic computation.

| Section | Content |
|----|----|
|  **1**  | Derivation of homogeneous transformation matrices that compose the FK chain |
|  **2**  | Derivation of end-effector pose using the ZXY Euler convention |
|  **3**  | Comparison of the analytical FK against RViz TF readback |
|  **4**  | Interactive workspace visualisation using RViz publishing |

**Assumptions and conventions**
- Angles are in radians unless explicitly stated.
- ZXY intrinsic Euler convention: `R = Rz(yaw) * Rx(pitch) * Ry(roll)`
- Frame chain: `world -> base -> shoulder -> upperarm -> lowerarm -> wrist -> gripper -> gripper_center`
- All link offsets are taken from the assignment description.
- The gripper DOF (joint 6) is not included in the FK and is set to 0 when commanding RViz.

In [1]:
# ---------------------------------------------------------------
# SETUP — imports and path resolution
# Run this cell first.  The notebook must be opened from its own
# directory:  edubot/assignment/Task1_Mobility_and_Workspace/
# ---------------------------------------------------------------

import numpy as np
import sympy as sp
from sympy.utilities.lambdify import lambdify

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# Project layout:
#   <project_root>/assignment/Task1_Mobility_and_Workspace/  <- This notebook
#   <project_root>/assignment/robot_kinematics.py            <- BayMax Kinematics Definition
#   <project_root>/ros_ws/src/assignment/assignment/         <- ROS2 nodes
_NOTEBOOK_DIR    = Path().r# Project layout:esolve()
_PROJECT_ROOT    = _NOTEBOOK_DIR.parent.parent
_KINEMATICS_DIR  = _PROJECT_ROOT / 'assignment'
_ROS_NODES_DIR   = _PROJECT_ROOT / 'ros_ws' / 'src' / 'assignment' / 'assignment'

for _p in [str(_KINEMATICS_DIR), str(_ROS_NODES_DIR)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from robot_kinematics import RobotKinematics

print(f'Project root   : {_PROJECT_ROOT}')
print(f'Kinematics dir : {_KINEMATICS_DIR}')
print(f'ROS nodes dir  : {_ROS_NODES_DIR}')
print('RobotKinematics imported OK.')

AttributeError: 'RcParams' object has no attribute '_get'

---
## Section 1 — Homogeneous Transformation Matrices

A **homogeneous transformation matrix** T is a 4x4 matrix that encodes both the rotation R (3x3 matrix) and a translation t (3x1 vector) in a single object between two reference reference frames:

```
T = | R  t |
    | 0  1 |
```

By multiplying two transforms `T_A_B` (B relative to A) and `T_B_C` (C relative to B) it is possible to obtain`T_A_C = T_A_B * T_B_C` (C relative to A).

Based on this observation, the reference frame of a given arm joint can be written relative to the reference frame of the previous joint in the arm chain using a compound of homogeneour transforms as follows:

```
T_[i-1][i] =  | I  t(dx, dy, dz) | * | Ri(ai)  0 | * | Rz(qi)  0 |
              | 0        1       |   |    0    1 |   |    0    1 | 
```
where:
- `t(dx,dy,dz)` positions the origin of joint i, in the coordinates frame of joint  i-1,
- `Ri(ai)` is a fixed frame-alignment rotation, (rotation of reference frame i relative to reference frame i-1)
- `Rz(qi)` is the actuated joint rotation (note that all joints rotate about their local Z-axis after frame alignment).

In [2]:
# ---------------------------------------------------------------
# Symbolic primitive transforms 
# All elements are SymPy symbols.
# ---------------------------------------------------------------

def sym_rot_x(theta):
    """4x4 homogeneous rotation about the X-axis by angle theta."""
    c, s = sp.cos(theta), sp.sin(theta)
    return sp.Matrix([
        [1,  0,  0, 0],
        [0,  c, -s, 0],
        [0,  s,  c, 0],
        [0,  0,  0, 1],
    ])


def sym_rot_y(theta):
    """4x4 homogeneous rotation about the Y-axis by angle theta."""
    c, s = sp.cos(theta), sp.sin(theta)
    return sp.Matrix([
        [ c, 0, s, 0],
        [ 0, 1, 0, 0],
        [-s, 0, c, 0],
        [ 0, 0, 0, 1],
    ])


def sym_rot_z(theta):
    """4x4 homogeneous rotation about the Z-axis by angle theta."""
    c, s = sp.cos(theta), sp.sin(theta)
    return sp.Matrix([
        [c, -s, 0, 0],
        [s,  c, 0, 0],
        [0,  0, 1, 0],
        [0,  0, 0, 1],
    ])


def sym_trans(x, y, z):
    """4x4 homogeneous pure translation by (x, y, z)."""
    return sp.Matrix([
        [1, 0, 0, x],
        [0, 1, 0, y],
        [0, 0, 1, z],
        [0, 0, 0, 1],
    ])


print('Example — Rotation around local Z-axis ')
t1 = sp.symbols('theta_1')
sp.pprint(sym_rot_z(t1))
print()
print('Example — Pure translation (tx, ty, tz):')
tx, ty, tz = sp.symbols('t_x t_y t_z')
sp.pprint(sym_trans(tx, ty, tz))

Example — Rotation around local Z-axis 
⎡cos(θ₁)  -sin(θ₁)  0  0⎤
⎢                       ⎥
⎢sin(θ₁)  cos(θ₁)   0  0⎥
⎢                       ⎥
⎢   0        0      1  0⎥
⎢                       ⎥
⎣   0        0      0  1⎦

Example — Pure translation (tx, ty, tz):
⎡1  0  0  tₓ ⎤
⎢            ⎥
⎢0  1  0  t_y⎥
⎢            ⎥
⎢0  0  1  t_z⎥
⎢            ⎥
⎣0  0  0   1 ⎦


### Kinematic chain — link by link

| Link transform | Translation (m) | Frame alignment | Joint DOF |
|---|---|---|---|
| T_world_base | (0, 0, 0) | Rz(pi) | — |
| T_base_shoulder | (0, -0.0452, 0.0165) | — | Rz(t1) |
| T_shoulder_upper | (0, -0.0306, 0.1025) | Ry(-pi/2) | Rz(t2) |
| T_upper_lower | (0.11257, -0.028, 0) | — | Rz(t3) |
| T_lower_wrist | (0.0052, -0.1349, 0) | Rz(+pi/2) | Rz(t4) |
| T_wrist_gripper | (-0.0601, 0, 0) | Ry(-pi/2) | Rz(t5) |
| T_gripper_center | (0, 0, 0.075) | — | — |

In [4]:
# ---------------------------------------------------------------
# Build consecutive reference frame transform symbolically.
# The product of all link transforms gives full transformation:
#   the pose of the gripper center expressed in the world frame.
# ---------------------------------------------------------------
t1, t2, t3, t4, t5 = sp.symbols('theta_1 theta_2 theta_3 theta_4 theta_5')

# TO: World     --- FROM: Base
T_world_base     = sym_trans(0, 0, 0)              * sym_rot_z(sp.pi)

#TO: Base      --- FROM: Shoulder (Rotation t1)
T_base_shoulder  = sym_trans(0, -0.0452, 0.0165)   * sym_rot_z(t1)

# TO: Shoulder  --- FROM: Upperarm (Rotation t2)
T_shoulder_upper = sym_trans(0, -0.0306, 0.1025)   * sym_rot_y(-sp.pi/2) * sym_rot_z(t2)

# TO: Upperarm  --- FROM: Lowerarm (Rotation t3)
T_upper_lower    = sym_trans(0.11257, -0.028, 0)    * sym_rot_z(t3)

# TO: Lowerarm  --- FROM: Wrist (Rotation t4)
T_lower_wrist    = sym_trans(0.0052, -0.1349, 0)    * sym_rot_z(sp.pi/2) * sym_rot_z(t4)

# TO: Wrist     --- FROM: Gripper
T_wrist_gripper  = sym_trans(-0.0601, 0, 0)         * sym_rot_y(-sp.pi/2) * sym_rot_z(t5)

# TO: Gripper     --- FROM: Gripper Center
T_gripper_center = sym_trans(0, 0, 0.075)

# Full kinematic chain
T_full = (T_world_base * T_base_shoulder * T_shoulder_upper
          * T_upper_lower * T_lower_wrist * T_wrist_gripper * T_gripper_center)

print('T_full[:3, 3]  — position vector of gripper center (symbolic):')
sp.pprint(T_full[:3, 3])

T_full[:3, 3]  — position vector of gripper center (symbolic):
⎡    -0.1351⋅(-sin(θ₁)⋅sin(θ₂)⋅sin(θ₃) + sin(θ₁)⋅cos(θ₂)⋅cos(θ₃))⋅cos(θ₄) + 0. ↪
⎢                                                                              ↪
⎢-0.1351⋅(sin(θ₂)⋅sin(θ₃)⋅cos(θ₁) - cos(θ₁)⋅cos(θ₂)⋅cos(θ₃))⋅cos(θ₄) + 0.1351⋅ ↪
⎢                                                                              ↪
⎣                                                 0.1351⋅(-sin(θ₂)⋅sin(θ₃) + c ↪

↪ 1351⋅(sin(θ₁)⋅sin(θ₂)⋅cos(θ₃) + sin(θ₁)⋅sin(θ₃)⋅cos(θ₂))⋅sin(θ₄) + 0.1349⋅si ↪
↪                                                                              ↪
↪ (-sin(θ₂)⋅cos(θ₁)⋅cos(θ₃) - sin(θ₃)⋅cos(θ₁)⋅cos(θ₂))⋅sin(θ₄) - 0.1349⋅sin(θ₂ ↪
↪                                                                              ↪
↪ os(θ₂)⋅cos(θ₃))⋅sin(θ₄) - 0.1351⋅(-sin(θ₂)⋅cos(θ₃) - sin(θ₃)⋅cos(θ₂))⋅cos(θ₄ ↪

↪ n(θ₁)⋅sin(θ₂)⋅sin(θ₃) + 0.0052⋅sin(θ₁)⋅sin(θ₂)⋅cos(θ₃) + 0.11257⋅sin(θ₁)⋅sin ↪
↪                                           

The above relationships are implemented in the class *RobotKinematics*, that can be found in the parent folder where this notebook is located.  

---
## Section 2 — End-Effector Pose: ZXY Euler Angles

The 4x4 transform `T_full` encodes both the **position** (last column) and **orientation** (top-left 3x3 rotation matrix R) of the gripper center relative to the world base frame.

```_symbolic_EE_pose
X = T_full[0, 3]
Y = T_full[1, 3]
Z = T_full[2, 3]
```

To express orientation the **ZXY intrinsic Euler convention** is used:

```
R = Rz(yaw) * Rx(pitch) * Ry(roll)
```

Using the symbolic rotation matrices defined previously, the defined rotation matrix is obtained.


In [ ]:
# ---------------------------------------------------------------
# Build ZXY intrinsic Euler Rotation Matrix, symbolically.
# ---------------------------------------------------------------
rot_x, rot_y, rot_z = sp.symbols('rx, ry, rz')

print('ZXY intrinsic Euler Rotation Matrix (symbolic):')
Rzxy = sym_rot_z(rot_z)*sym_rot_y(rot_y)*sym_rot_x(rot_x)
sp.pprint(Rzxy)


ZXY intrinsic Euler Rotation Matrix (symbolic):
⎡cos(ry)⋅cos(rz)  sin(rx)⋅sin(ry)⋅cos(rz) - sin(rz)⋅cos(rx)  sin(rx)⋅sin(rz) +
⎢                                                                             
⎢sin(rz)⋅cos(ry)  sin(rx)⋅sin(ry)⋅sin(rz) + cos(rx)⋅cos(rz)  -sin(rx)⋅cos(rz) 
⎢                                                                             
⎢   -sin(ry)                   sin(rx)⋅cos(ry)                            cos(
⎢                                                                             
⎣       0                             0                                       

 sin(ry)⋅cos(rx)⋅cos(rz)   0⎤
                            ⎥
+ sin(ry)⋅sin(rz)⋅cos(rx)  0⎥
                            ⎥
rx)⋅cos(ry)                0⎥
                            ⎥
   0                       1⎦
⎡             cos(ry)⋅cos(rz)                            -sin(rz)⋅cos(ry)     
⎢                                                                             
⎢sin(rx)⋅sin(ry)⋅cos(rz) + sin

In [6]:
from scipy.spatial.transform import Rotation as R
# Create a rotation object (e.g., ZYX sequence, 90 deg z, 0 deg y, 0 deg x)
r = R.from_euler('XYZ', [90, 0, 0], degrees=True)
# Convert to 3x3 matrix
matrix = r.as_matrix()
print(matrix)

[[ 1.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  2.22044605e-16 -1.00000000e+00]
 [ 0.00000000e+00  1.00000000e+00  2.22044605e-16]]


Therefore, using the full transformation matrix defined in section 1, the Euler angles of the EE can be obtained using the relationships below.

```
pitch  (theta,  about X)  =  asin(R[2,1])
roll   (phi, about X)     =  atan2( -R[2,0],  R[2,2])
yaw    (psi,   about Z)   =  atan2( -R[0,1],  R[1,1])
```

These formulas are valid as long as pitch is not exactly +/-90 deg (gimbal-lock singularity). The relations deduced are implemented in the *RobotKinematics* class. The symbolic forward kinematics can be obtained by running the cell below.



In [10]:
# ---------------------------------------------------------------
# BayMax forward Kinematics
# ---------------------------------------------------------------

robot = RobotKinematics()

sym_pos = robot._symbolic_EE_pose[0:2]
sym_ori = robot._symbolic_EE_pose[3:6]

print('EE pose vector components [x, y, z, pitch, roll, yaw]')
print('Symbolic Position:')
print()
sp.pprint(sym_pos)
print()
print('Symbolic Orientation:')
sp.pprint(sym_ori)
print()


EE pose vector components [x, y, z, pitch, roll, yaw]
Symbolic Position:

[-0.1351⋅(-sin(θ₁)⋅sin(θ₂)⋅sin(θ₃) + sin(θ₁)⋅cos(θ₂)⋅cos(θ₃))⋅cos(θ₄) + 0.1351 ↪

↪ ⋅(sin(θ₁)⋅sin(θ₂)⋅cos(θ₃) + sin(θ₁)⋅sin(θ₃)⋅cos(θ₂))⋅sin(θ₄) + 0.1349⋅sin(θ₁ ↪

↪ )⋅sin(θ₂)⋅sin(θ₃) + 0.0052⋅sin(θ₁)⋅sin(θ₂)⋅cos(θ₃) + 0.11257⋅sin(θ₁)⋅sin(θ₂) ↪

↪  + 0.0052⋅sin(θ₁)⋅sin(θ₃)⋅cos(θ₂) - 0.1349⋅sin(θ₁)⋅cos(θ₂)⋅cos(θ₃) - 0.028⋅s ↪

↪ in(θ₁)⋅cos(θ₂) - 0.0306⋅sin(θ₁), -0.1351⋅(sin(θ₂)⋅sin(θ₃)⋅cos(θ₁) - cos(θ₁)⋅ ↪

↪ cos(θ₂)⋅cos(θ₃))⋅cos(θ₄) + 0.1351⋅(-sin(θ₂)⋅cos(θ₁)⋅cos(θ₃) - sin(θ₃)⋅cos(θ₁ ↪

↪ )⋅cos(θ₂))⋅sin(θ₄) - 0.1349⋅sin(θ₂)⋅sin(θ₃)⋅cos(θ₁) - 0.0052⋅sin(θ₂)⋅cos(θ₁) ↪

↪ ⋅cos(θ₃) - 0.11257⋅sin(θ₂)⋅cos(θ₁) - 0.0052⋅sin(θ₃)⋅cos(θ₁)⋅cos(θ₂) + 0.1349 ↪

↪ ⋅cos(θ₁)⋅cos(θ₂)⋅cos(θ₃) + 0.028⋅cos(θ₁)⋅cos(θ₂) + 0.0306⋅cos(θ₁) + 0.0452]

Symbolic Orientation:
⎡     ⎛                                                                        ↪
⎢     ⎜                                                                        ↪
⎣atan2⎝

In order to allow fast numerical evaluation of the robot forward kinematics given its joint angles, lambdify function is used. The numerical forward kinematics can be acessed trough method robot.forward_kinematics.

---
## Section 3 — Comparison Analytical Forward Kinematics with RViz

For this section, two helper ROS2 nodes are used from the `assignment` ROS2 package, included under ros_ws/src/assignment:

- **`JointStateSetter`** (`set_joint_conf.py`): publishes joint angles to `joint_states`, which drives `robot_state_publisher` and updates RViz.
- **`EEPositionReader`** (`read_ee_pose.py`): looks up the `world → gripper_center` TF and returns `[x, y, z, pitch, roll, yaw]`.

To test the implemented FW kinematics relations, the following preocess is followed:

1. Define the test joint configurations (The gripper DOF (joint 6) and is set to 0 when using Rviz)
2. Compute FK numerically using `robot.forward_kinematics` (Section 2).
4. Publish the joint angles to RViz (using JointStateSetter) and wait for TF to update.
4. Read back the EE pose from TF (using EEPositionReader).


**Prerequisites — run in a separate terminal before executing the rest of this code:**

```bash
cd ~/edubot/ros_ws
source /opt/ros/humble/setup.bash
source install/setup.bash
ros2 launch lerobot rviz.launch.py
```

In [ ]:
# ---------------------------------------------------------------
# Check whether robot_state_publisher is already running.
# If not, auto-launch it (without RViz GUI).
# ---------------------------------------------------------------
import subprocess
import time

_rsp_proc = None  # handle for auto-launched process (used for cleanup)


def _rsp_running():
    try:
        r = subprocess.run(['ros2', 'node', 'list'],
                           capture_output=True, text=True, timeout=5)
        return 'robot_state_publisher' in r.stdout
    except Exception:
        return False


if _rsp_running():
    print('[OK] robot_state_publisher is already running.')
else:
    print('[INFO] robot_state_publisher not found — launching it now...')
    _rsp_proc = subprocess.Popen(
        ['ros2', 'launch', 'lerobot', 'rviz.launch.py'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for _ in range(30):
        time.sleep(0.5)
        if _rsp_running():
            print('[OK] robot_state_publisher is running.')
            break
    else:
        print('[WARNING] robot_state_publisher did not start in 15 s.')
        print('          Run manually: ros2 launch lerobot rviz.launch.py')

In [ ]:
# ---------------------------------------------------------------
# Initialise ROS2, import the assignment-package helpers, and
# warm up the TF buffer so lookups succeed immediately.
# ---------------------------------------------------------------
import rclpy
from set_joint_conf import JointStateSetter
from read_ee_pose   import EEPositionReader

if not rclpy.ok():
    rclpy.init()

_ros_node = rclpy.create_node('fw_kinematics_notebook')
_setter   = JointStateSetter(_ros_node)
_reader   = EEPositionReader(_ros_node)

# Spin briefly to populate the TF buffer
# (tf_static transforms are published once at startup)
print('Warming up TF buffer...', end=' ', flush=True)
for _ in range(30):
    rclpy.spin_once(_ros_node, timeout_sec=0.1)
print('done.')

In [ ]:
# ---------------------------------------------------------------
# For each test configuration:
#   1. Compute FK analytically (notebook derivation, Phase 2)
#   2. Set joints in RViz via joint_states publisher
#   3. Read back EE pose from TF
#   4. Compare xyz positions
# ---------------------------------------------------------------

# Format: [t1, t2, t3, t4, t5, gripper=0]  — all in radians
TEST_CONFIGS = [
    [0.0,  0.0,  0.0,  0.0,  0.0, 0.0],
    [0.5,  0.3, -0.3,  0.2,  0.0, 0.0],
    [1.0, -0.5,  0.4,  0.3,  1.0, 0.0],
    [-0.8, 0.4,  0.2, -0.4,  0.5, 0.0],
]

results = []
for joints in TEST_CONFIGS:
    q = joints[:5]
    # Analytical FK
    fk = np.array(_ee_pose_num(*q)).flatten()
    # Publish to RViz
    _setter.set(joints)
    # Read back from TF
    tf_pose = _reader.read(retries=30)
    results.append((q, fk, tf_pose))

# ------ Print comparison table ------
print(f'{"Config [rad]":<36}  {"FK xyz (m)":<30}  {"TF xyz (m)":<30}  Pos ok?')
print('-' * 112)
for q, fk, tf in results:
    fk_s = f'{fk[0]:+.4f}  {fk[1]:+.4f}  {fk[2]:+.4f}'
    if tf is not None:
        tf_s  = f'{tf[0]:+.4f}  {tf[1]:+.4f}  {tf[2]:+.4f}'
        match = 'YES' if np.allclose(fk[:3], tf[:3], atol=1e-3) else 'NO'
    else:
        tf_s  = 'TF lookup failed              '
        match = 'N/A'
    qs = str([round(v, 2) for v in q])
    print(f'{qs:<36}  {fk_s:<30}  {tf_s:<30}  {match}')

In [ ]:
# ---------------------------------------------------------------
# Clean up the ROS2 node.
# The robot_state_publisher is kept alive for Phase 4.
# ---------------------------------------------------------------
_ros_node.destroy_node()
rclpy.shutdown()
print('ROS2 node destroyed and context shut down.')
if _rsp_proc is not None:
    print('robot_state_publisher auto-launched — keeping it alive for Phase 4.')

---
## Phase 4 — Workspace Visualisation

The **reachable workspace** is the set of all end-effector positions achievable by sweeping joint angles within their limits.

The interactive controls below let you:
1. **Select which joints to sweep** — unchecked joints are held fixed at 0 rad.
2. **Set the swept range** for each joint (in degrees).
3. Choose the **resolution** (samples per active joint).
4. Click **Compute & Plot** to generate the 3D scatter in this notebook.
5. Click **Publish to RViz** to stream the workspace point cloud to `/fk_ee` in RViz.

> **Note:** `n_active_joints ^ resolution` points are computed.  
> With 5 active joints at resolution 12, this is 12^5 = 248 832 points — fast with lambdify.
> At resolution 20 it becomes 20^5 = 3.2 M — may take ~30 s.

In [ ]:
# ---------------------------------------------------------------
# Build the interactive widget UI.
# Run this cell first, then the next two cells to register the
# button callbacks, then interact with the controls.
# ---------------------------------------------------------------
import ipywidgets as widgets
from IPython.display import display

_JOINT_LABELS     = [
    'theta1  Shoulder Rotation',
    'theta2  Shoulder Pitch',
    'theta3  Elbow',
    'theta4  Wrist Pitch',
    'theta5  Wrist Roll',
]
_JOINT_BOUNDS_DEG = [(-115, 115), (-90, 90), (-90, 90), (-90, 90), (-180, 180)]

# Checkbox: which joints to sweep
_checks = [
    widgets.Checkbox(value=True, description=lbl,
                     layout=widgets.Layout(width='260px'))
    for lbl in _JOINT_LABELS
]

# Range slider: joint range in degrees
_range_sliders = [
    widgets.FloatRangeSlider(
        value=[lo, hi], min=lo, max=hi, step=5.0,
        description='range (deg):', readout_format='.0f',
        style={'description_width': '90px'},
        layout=widgets.Layout(width='380px'),
    )
    for (lo, hi) in _JOINT_BOUNDS_DEG
]

# Resolution: samples per active joint
_res_slider = widgets.IntSlider(
    value=12, min=5, max=20, step=1,
    description='Resolution:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='380px'),
)

_compute_btn = widgets.Button(
    description='Compute & Plot',
    button_style='primary',
    layout=widgets.Layout(width='160px'),
)
_rviz_btn = widgets.Button(
    description='Publish to RViz',
    button_style='success',
    layout=widgets.Layout(width='160px'),
)
_out = widgets.Output()

_rows = [widgets.HBox([c, s]) for c, s in zip(_checks, _range_sliders)]
_ui = widgets.VBox([
    widgets.HTML('<b>Workspace configuration</b><br>'
                 'Select which joints to sweep and set their ranges.'),
    *_rows,
    _res_slider,
    widgets.HBox([_compute_btn, _rviz_btn]),
    _out,
])
display(_ui)

# Shared storage for the latest computed workspace
_ws = {'x': None, 'y': None, 'z': None, 'label': ''}

In [ ]:
# ---------------------------------------------------------------
# Callback: compute workspace from _ee_pose_num and plot 3D scatter.
# _ee_pose_num is vectorised (via lambdify), so the full grid is
# evaluated in one NumPy call.
# ---------------------------------------------------------------

def _compute_workspace():
    res = _res_slider.value
    spaces = []
    for i in range(5):
        if _checks[i].value:
            lo_deg, hi_deg = _range_sliders[i].value
            spaces.append(np.linspace(np.deg2rad(lo_deg), np.deg2rad(hi_deg), res))
        else:
            spaces.append(np.array([0.0]))
    grids = np.meshgrid(*spaces, indexing='ij')
    flat  = [g.flatten() for g in grids]
    result = np.array(_ee_pose_num(*flat)).reshape(6, -1)
    return result[0], result[1], result[2]


def _on_compute(btn):
    with _out:
        _out.clear_output(wait=True)
        active_labels = [_JOINT_LABELS[i] for i, c in enumerate(_checks) if c.value]
        if not active_labels:
            print('[WARNING] No joints selected — check at least one box.')
            return
        n_pts = _res_slider.value ** len(active_labels)
        print(f'Sweeping {len(active_labels)} joints at resolution {_res_slider.value} '
              f'=> {n_pts:,} points  ...')
        x, y, z = _compute_workspace()
        _ws['x'], _ws['y'], _ws['z'] = x, y, z
        _ws['label'] = ', '.join(active_labels)
        print(f'{len(x):,} points computed.')

        fig = plt.figure(figsize=(8, 6))
        ax  = fig.add_subplot(111, projection='3d')
        sc  = ax.scatter(x, y, z, s=1, alpha=0.25, c=z, cmap='viridis')
        plt.colorbar(sc, ax=ax, label='z (m)', shrink=0.6)
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_zlabel('z (m)')
        ax.set_title(f'Reachable workspace\nsweeping: {_ws["label"]}')
        plt.tight_layout()
        plt.show()


_compute_btn.on_click(_on_compute)

In [ ]:
# ---------------------------------------------------------------
# Callback: publish the computed workspace to /fk_ee in RViz.
# Uses transient-local QoS so RViz receives the marker even if
# it subscribes after this cell runs.
#
# In RViz: Add > By topic > /fk_ee > Marker  (or Add > Marker
# and set the topic manually to /fk_ee).
# ---------------------------------------------------------------

def _on_publish_rviz(btn):
    with _out:
        if _ws['x'] is None:
            print('[ERROR] No workspace data — click Compute & Plot first.')
            return

        import rclpy
        from rclpy.qos import (QoSProfile, DurabilityPolicy,
                                ReliabilityPolicy, HistoryPolicy)
        from visualization_msgs.msg import Marker
        from geometry_msgs.msg import Point
        from builtin_interfaces.msg import Duration

        if not rclpy.ok():
            rclpy.init()

        _pub_node = rclpy.create_node('workspace_pub_notebook')
        _qos = QoSProfile(
            depth=1,
            durability=DurabilityPolicy.TRANSIENT_LOCAL,
            reliability=ReliabilityPolicy.RELIABLE,
            history=HistoryPolicy.KEEP_LAST,
        )
        _pub = _pub_node.create_publisher(Marker, '/fk_ee', _qos)

        # Build the POINTS marker
        marker = Marker()
        marker.header.frame_id = 'world'
        marker.ns              = 'workspace'
        marker.id              = 0
        marker.type            = Marker.POINTS
        marker.action          = Marker.ADD
        marker.scale.x         = 0.004   # point width in metres
        marker.scale.y         = 0.004
        marker.color.r         = 0.0
        marker.color.g         = 0.75
        marker.color.b         = 1.0
        marker.color.a         = 0.4
        marker.lifetime        = Duration(sec=0, nanosec=0)  # never expire
        marker.points = [
            Point(x=float(xi), y=float(yi), z=float(zi))
            for xi, yi, zi in zip(_ws['x'], _ws['y'], _ws['z'])
        ]
        marker.header.stamp = _pub_node.get_clock().now().to_msg()

        # Publish a few times to make sure late-joining RViz receives it
        for _ in range(5):
            _pub.publish(marker)
            rclpy.spin_once(_pub_node, timeout_sec=0.1)

        _pub_node.destroy_node()
        rclpy.shutdown()
        print(f'Published {len(_ws["x"]):,} workspace points to /fk_ee.')
        print('In RViz: Add > Marker > Topic = /fk_ee')


_rviz_btn.on_click(_on_publish_rviz)